# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型  |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** どこから読み、最初に何をするか |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# コンテナの更新と追加

**当てるのは人。機械は知らせるだけ。** 上げる前に必ず控えを取る。上げたら**画面で確かめる**。
対象は本番だけ(検証機は 2026-09-07 に退役)。**試す場所は無い**という前提で読む。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 0. まとめて進める(`update-containers.ps1`。2026-10-16〜)

**普段はこれ 1 本でよい。** 番号を選ぶと、下の §3〜§7 の手順を決まった順に進める(利用者の指示「Node と Logto 等の上げ方を簡略化。選択肢を設けて実行」)。

| 番号 | すること | 本番が変わるか |
|---|---|---|
| 1 | 更新を調べる(ホストの `check-updates.sh`。§4) | 変わらない |
| 2 | **修正版を当てる** —— PHP(web の土台)・MariaDB・PostgreSQL・Logto 以外の像など。一覧から番号で選ぶ(Enter で ★ の全部)。**版(タグ)はそのまま** | `yes` のあと変わる |
| 3 | **Logto を次の版へ**。GitHub の正式版から次の 1 つを出し、リリースノートの URL を見せる(§5-2) | `yes` を 2 回(配備・移行) |
| 4 | いまの様子(`docker compose ps`・Logto のログ) | 変わらない |

2 と 3 の中でやること(**この順から外れない**):

1. ホストでレジストリに**いまの digest** を聞く(check-updates.sh と同じ `imagetools`)
2. 手元の `compose.yaml` / `compose.vps.yaml` / `docker/*/Dockerfile` の**その 1 行だけ**を `名前:タグ@sha256:…` に書き換える(改行・BOM はそのまま)
3. `check.php` を通す。**落ちたら書き換えを元に戻して止まる**
4. `yes` と打つ → `deploy-to-host.ps1 -Action deploy -Backup -Yes`(**控えは配備の前**。取れなければ何も送らずに止まる。GitHub の控えも出る)
5. 2: `docker compose pull`(image のもの)/ `docker compose build --pull`(web)→ 全体の `up -d`
   3: `pull logto` → 移行の一覧 → **もう一度 `yes`** → 移行を新しい像で当てる → 全体の `up -d` → ログ
6. §7 の見るものを出す(画面で確かめるのは人)

**間違っていたら止まる・揺れはやり直す**(2026-10-16〜):

| いつ | 何を見るか | 駄目なとき |
|---|---|---|
| 配備の前(`yes` より前) | 手元: `image:` / `FROM` が `名前[:タグ]@sha256:64桁` の形か | **何も送らずに止まる。** ファイルを直して `r` で確かめ直す。Enter でやめる(手元を戻す) |
| 〃 | ホスト: 手元の compose を一時フォルダへ写し、本番の `.env` で `docker compose config -q`(置き場は汚さない) | 〃 |
| 〃 | ホスト: 書いてある像が全部レジストリにあるか(`imagetools inspect`) | 〃 |
| レジストリへの問い合わせ・`pull`・`build` | 失敗したら 10 秒・20 秒待って**自動でやり直す**(計 3 回) | `up -d` せずに止まる。`b` で手元と本番の compose を元に戻す |
| `up -d` のあと | 全部が起動したか(restarting・unhealthy・0 以外で終了)。starting の間は待つ(3 分。Logto は 5 分) | ログを出して止まる。`b` で元の版に戻して立て直す |

**Logto の移行だけはやり直さない**(1 回だけ)。移行のあとに起動しなかったときも自動では戻さない(版を戻すだけでは直らない。§6 か控えからの復元)。
移行の前に止めたときは、手元と本番の compose を本番で動いている版へ戻す(戻さないと、次の `up -d` で移行なしに新しい像が起動する)。

上げないもの: **タグの無い固定**(`nginx@sha256:…`・mailpit・certbot・postfix)は上げ先の版が分からないので一覧の下に名前だけ出す。**メジャー版**(postgres 17 → 18 など)はタグを変えるので扱わない。
Logto の版を**飛ばす**ときは `skip` と打たせる(移行を飛ばすと戻せない)。

流れを見るだけなら `-DryRun`(繋がない・書き換えない・配備しない。digest は仮の値)。1 つだけなら `-Choice check|refresh|logto|status`。

🔴 **本番が変わる**(番号 2・3)—— 別の窓で開きます。途中で `yes` の入力を求めます。


In [ ]:
%%terminal
.\update-containers.ps1

## 1. いまの形

```
プロジェクト  kosenmap        ← 2026-09-07 に Test から改名
コンテナ      km-*            ← 同上。dev-* から改名
ボリューム    test_*          ← 実体名は据え置き(compose の volumes: の name: で固定)
```

`.env` に `COMPOSE_FILE="compose.yaml:compose.vps.yaml"` が入っているので、**ホストでは素の `docker compose` で両方が読まれる。**

| サービス | image | 固定の仕方 |
|---|---|---|
| `logto` / `postgres` / `mariadb` / `glances` | `名前:タグ@sha256:…`(例 `ghcr.io/logto-io/logto:1.43.0@sha256:…`) | **タグと digest の両方**(2026-09-15)。動くのは digest。タグは `check-updates.sh` が版番号と「固定より新しい修正版」を読むために残す |
| `reverse-proxy` / `mailpit` / `certbot` / `mailserver` | `…@sha256:…` | **digest** |
| `web` | `build:`(自前) | Dockerfile の `FROM` を `タグ@digest` で固定。ビルド元の更新は `base` として知らせる。**compose に digest を貼らない** |

**`:latest` は1つも無い**(`check.php` の `compose` が見張る)。

**本番では `mailpit` を起動しない**(`compose.vps.yaml` の `profiles`)。上限・利用者の決まりは §1-1。

**2026-11-02 に `soketi`(WebSocket)を外した**(§1-2)。外したサービスのコンテナは、配備の `docker compose up -d --remove-orphans`(`-Action up`)で消える。
`phpmyadmin` も同じ日に外したが、**11-03 にホストの 127.0.0.1:8281 だけで戻した**(§1-2。SSH のトンネルの先で開く)。

### いまの像

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker ps --format '{{.Names}}\t{{.Image}}\t{{.Status}}' | sort

### 1-1. 上限・起動しないもの・利用者(2026-09-14 の診断で追加)

**コンテナごとの上限。** それまでどのサービスにも上限が無く、1つの暴走(PHP のメモリ・fork の連鎖)が**ホスト全体(RAM 2GB)を道連れにできた。**
本番の実測の 2〜4 倍を目安に置いている。全員が同時に使い切る前提の枠ではなく、**1つが暴れても他を巻き込まない**ための天井。

| サービス | `mem_limit` | `pids_limit` | `no-new-privileges` |
|---|---|---|---|
| `logto` | 512m | 256 | あり |
| `mariadb` | 512m | 200 | あり |
| `web` | 384m | 200 | あり |
| `postgres` | 256m | 200 | あり |
| `reverse-proxy` / `mailpit` / `certbot` | 128m | 200 | あり |
| `mailserver` | 192m | 200 | **付けない**(postfix の `postdrop` が setgid で動くので、付けると送信が止まる) |

`certbot` と `mailserver` の分は `compose.vps.yaml`、残りは `compose.yaml`(VPS にもそのまま効く)。
`no-new-privileges` は setuid で権限を上げる道を塞ぐだけで、root で起動してから一般利用者へ下りる形(Apache・nginx・gosu)は妨げない。

**`web` は Apache のワーカーが最大 150 のまま**なので、重い PHP が並ぶと 384m に当たりうる。配備後の数日は下のセルで `OOMKilled` と使用量を見る。
当たるなら `docker/php/apache-km.conf` で `MaxRequestWorkers` を絞るか、`mem_limit` を上げる。`mem=0` / `pids=0` は上限なし(配備前の形)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker inspect --format '{{.Name}}  mem={{.HostConfig.Memory}}  pids={{.HostConfig.PidsLimit}}  {{.HostConfig.SecurityOpt}}  OOMKilled={{.State.OOMKilled}}' $(docker compose ps -q) | sed 's#^/##' | sort
echo
docker stats --no-stream --format '{{.Name}}\t{{.MemUsage}}\t{{.PIDs}}' | sort

**本番では Mailpit を起動しない。** 本番の送信は `mailserver` が担い、Mailpit は1通も受けていなかった(実測 0 通)のに、無認証の画面を持つサービスが devnet に居続けていた。
`compose.vps.yaml` で `profiles: ["mailpit"]` を付けたので、`docker compose up -d` では作られない(`reverse-proxy` も Mailpit を待たない)。

- nginx の 8025 の受け口は Mailpit が居なくても起動する。**本番では 2026-09-18 から publish しないので、外からは繋がらない**(管理画面のサービス一覧のリンクも本番では開かない)
- 検証で一時的に使うなら `docker compose --profile mailpit up -d mailpit`。済んだら下のセルで片付ける
- **既に動いているコンテナは `up` しても消えない。** 配備のあとに一度だけ下のセルを流す(メールは 0 通なので失うものは無い)
- 校内 LAN の構成(`compose.yaml` 単体)では従来どおり起動する

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Mailpit のコンテナを止めて消します(本番では使っていません。メールは 0 通)"
docker compose --profile mailpit stop mailpit && docker compose --profile mailpit rm -f mailpit
docker ps --format '{{.Names}}' | grep -x km-mailpit || echo "km-mailpit は居ません"

### 1-2. Soketi は外した・phpMyAdmin はトンネルの先だけで戻した(2026-11-02 / 11-03)

| もの | 何に使っていたか | いま |
|---|---|---|
| `soketi`(6001) | 管理画面のチャット(本番で 0 件)と、サービス監視のリアルタイム表示 | **外した**(11-02)。サービスの生死は「タスクマネージャー」の「サービス」タブ(30 秒ごとに `admin/api/health-check.php`) |
| `phpmyadmin` | MariaDB を画面で見る | 11-02 に外し、**11-03 に戻した**(利用者の指示)。**ホストの 127.0.0.1:8281 だけで待つ**(nginx は通さない)。開き方は §5 の「Logto Console と phpMyAdmin を開く」 |
| テーブルの外部接続 API(`/api/table.php`) | 外から DB の表を読む・書く(本番でキー 0 件) | 外した(11-02) |

- 11-02 の脆弱性のメール(15 件)はすべて Soketi の部品だった。外したので、勧告・コンテナ・`/ws/app/` の入口がまとめて消えた
- 外したファイルは `Old/removed-2026-11-02/`。phpMyAdmin の設定(`docker/phpmyadmin/config.user.inc.php`。root と空パスワードの禁止・30 分・環境保管領域)は 11-03 にそこから戻した
- **phpMyAdmin の決まり**(compose.yaml の phpmyadmin):
  - `ports:` は `127.0.0.1:8281:80` だけ。インターネットからは届かない。**ログインは MariaDB の利用者**(nginx のゲートは通らない)。root は既定で禁止(`.env` の `KM_PMA_ALLOW_ROOT=1` のときだけ)
  - 網は `appdb`(MariaDB へ。利用者の Host は `172.30.2.%`)と `edge`(ホストへ publish するため。`appdb` は internal で publish できない)
  - 環境保管領域(`pma` 利用者・`phpmyadmin` の DB・`.env` の `KM_PMA_CONTROLPASS`)とボリューム `test_phpmyadmin_sessions` は、11-02 に消さずに残していたので**そのまま使う**。**消さないこと**(11-02 にあった「名残を消す」セルは 11-03 に外した)
  - 以前の手順(root 禁止の経緯・環境保管領域の作り方・設定を保存したら TypeError)は `Old/removed-2026-11-02/docs/05-containers-soketi-phpmyadmin.ipynb`
- DB に残してあるもの: `km_chat_*` の表(0 件)・`km_table_api_keys`(0 件)。`.env` の `SOKETI_*` の行はもう読まれない

## 2. なぜ自動で当てないのか

| | 自動でよいか | なぜ |
|---|---|---|
| Ubuntu のセキュリティ更新 | **よい**(unattended-upgrades) | ただし再起動は自動にしない |
| **Logto** | **駄目** | 起動時に DB の移行が走る。**Logto は管理画面のゲートそのもの**なので、落ちると直しに行く手段ごと失う |
| MariaDB / PostgreSQL | **駄目** | データの入れ物。戻せない |
| nginx / certbot / postfix / mailpit | 人が見てから | 入口・証明書・メールが止まる |

## 3. 版を固定する(`--pins`)

ホストで出した `image:` 行を `compose.yaml` / `compose.vps.yaml` へ貼る。**`web` / `logto` は貼らない**(出てこない)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
./scripts/check-updates.sh --path /opt/kosenmap --pins

## 4. 更新を見つける

毎日 4:17 に調べて、変化があればメールが来る。手で見るならこのセル。

| 種類 | 何が分かるか | 打つ手 |
|---|---|---|
| `release` | 新しい版が公開された(Logto) | 上げるか判断する(§5-2) |
| `digest` | 動くタグの中身が入れ替わった | 気づかないうちに上がりうる。固定する |
| `series` | 系列のタグ(`11.4` など)の中身が入れ替わった = **修正版** | 控え → `pull` → `up -d`(§5-3) |
| `base` | 自前ビルドのビルド元が入れ替わった | 控え → `build --pull` → `up -d`(§5-4)。**pull では当たらない** |

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
./scripts/check-updates.sh --path /opt/kosenmap

## 5. 当て方

### 5-1. 共通の手順(この順番から外れない)

1. **控えを取る**([03-backup](03-backup.ipynb) の「いま控えを取る」)。`summary.txt` の行数を見る
2. 手元の compose を書き換える。**版は必ず具体的な値**(`latest` に戻さない)
3. `check.php` を通す([01-daily-check](01-daily-check.ipynb))
4. 配備する。**compose を変えたら `-Action up`**([02-deploy](02-deploy.ipynb))
5. **§7 の一覧を画面で確かめる**

### 5-3. 修正版(series)を当てる

compose は書き換えない(タグは同じ)。**先に控えを取ってから。** `SERVICES` を当てるものに書き換える。
**メジャー版は上げない**(`postgres:17` → `18` はデータ領域の形式が変わり、そのままでは起動しない)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "指定したサービスの像を取り直して入れ替えます(先に控えを取りましたか?)" --timeout 900
SERVICES="node"
docker compose pull $SERVICES
docker compose up -d
docker compose ps

### 5-4. ビルド元(base)を当てる

`web` は自前でビルドする。**pull では変わらない。** `src/` は bind mount なので、PHP のコードだけならビルドは要らない。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "web をビルドし直して入れ替えます(先に控えを取りましたか?)" --timeout 1800
docker compose build --pull web
docker compose up -d
docker compose ps

### 5-2. Logto(一段重い)

- **1つずつ上げる**(飛ばさない)。**会期中に上げない**。リリースノートの breaking change を読む
- 上げた直後に**サインインを実際に通す**(管理画面と Android の両方)
- **`docker compose up -d logto` だけを叩かない**(`depends_on` が評価されない)。全体の `up -d` にする

手順:

1. **控えを取る**(飛ばすと戻れない)
2. 手元の `compose.yaml` の logto を新しい版にする(digest にしない)
3. `check.php` → **`-Action deploy`**(`up` にしない。先に移行を当てるので、ここではファイルを置くだけ)
4. 新しい像を取る → 5. 移行を**新しい像で**当てる → 6. 立て直す → 7. 起動を見届ける → 8. §7 を確かめる

**`NEW` を上げる先の版に書き換えてから**、4 から順に実行する。

### 4. 新しい像を取る

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto の新しい像を取ります(まだ入れ替えません)" --timeout 900
docker compose pull logto

### 5a. 移行の中身を見る

使い捨てのコンテナを新しい像で立てて、**当てずに一覧だけ**出す。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --confirm "使い捨ての Logto コンテナを立てて、移行の一覧を出します(DB は書き換えません)" --timeout 900
NEW=1.44.0
docker compose run --rm --entrypoint sh logto -c "cd /etc/logto && npm run alteration list $NEW"

### 5b. 移行を当てる

動いている古いコンテナで叩いても、新しい移行は入っていないので何も起きない。**新しい像で**走らせる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto の DB 移行を当てます。控えは取りましたか?(戻すのは難しい)" --timeout 1800
NEW=1.44.0
docker compose run --rm --entrypoint sh logto -c "cd /etc/logto && npm run alteration deploy $NEW"

### 6. 立て直す

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "docker compose up -d で Logto を新しい版に入れ替えます" --timeout 900
docker compose up -d

### 7. 起動を見届ける

`migration` の行が並んで終わっていれば通っている。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose logs --tail 80 logto

#### Logto Console と phpMyAdmin を開く(2026-11-02〜。11-03 から SOCKS のトンネル)

どちらも**ホストの 127.0.0.1 にだけ**出ている(`compose.yaml` の logto の `127.0.0.1:3002`・phpmyadmin の `127.0.0.1:8281`)。nginx の入口は無い。

**kmops の鍵(`km_ops`)では開けない。** ホストの `~kmops/.ssh/authorized_keys` の鍵には `no-port-forwarding` が付いている
(降格のときに `host-ops-user.sh` が付けた)。`-L` でも `-D` でも待ち受けは開くが、先へは「administratively prohibited」で断られる(11-03 に実測)。
**`km` の鍵(`km_vps`)で SOCKS のトンネル(`-D`)を張り、そのトンネルを通す Chrome で開く。**

いちばん簡単なのはデスクトップの **`トンネル開通SSH.cmd`**:
トンネルを別の窓で開く(開いていれば使い回す)→ 専用のプロフィールの Chrome を SOCKS5 指定で起動して 2 つを開く → **その Chrome を閉じるとトンネルも切る**。

手で張るなら(窓を 2 つ):

```powershell
ssh -N -D 127.0.0.1:10800 -o ExitOnForwardFailure=yes -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp
& "C:\Program Files\Google\Chrome\Application\chrome.exe" --user-data-dir="$env:LOCALAPPDATA\KosenMap\chrome-socks" --proxy-server="socks5://127.0.0.1:10800" --proxy-bypass-list="<-loopback>" http://localhost:3002 http://localhost:8281
```

- **`--proxy-bypass-list="<-loopback>"` が要る。** 無いと Chrome は localhost をプロキシに通さず、手元の PC の 3002 を見に行く
- **プロフィールを分ける**(`--user-data-dir`)。ふだんの Chrome が起動していると、同じプロフィールでは引数が無視される
- Console の戻り先(サインインのあとに戻る URL)は Logto が `ADMIN_ENDPOINT` から作る。いまは `http://localhost:3002`(`.env` の `LOGTO_ADMIN_ENDPOINT` で変えられる)。**DB は書き換えていない**
- phpMyAdmin のログインは MariaDB の利用者。URL は `PMA_ABSOLUTE_URI`(既定 `http://localhost:8281/`)
- 学校の回線は 22 番を通さないので、自宅などから張る。管理画面の「設定」にも同じコマンドが出る

**開けないとき**(サインインのあとに真っ白・戻り先の誤り など):

1. トンネルの窓に `open failed` が出ていないか(`administratively prohibited` なら鍵が違う。`Connection refused` ならその番号で誰も待っていない)
2. `docker compose logs --tail 80 logto`(か `phpmyadmin`)と、ホストで `curl -sS -o /dev/null -w '%{http_code}\n' http://127.0.0.1:3002/`(`8281`)を見る
3. Console がそれでも駄目なら**前の形に戻す**: `Old/removed-2026-11-02/compose.vps.yaml.before` と `default.conf.template.before` にある logto の 3002 の publish と nginx の 3002 の server を戻し、
   `.env` の `LOGTO_ADMIN_ENDPOINT` を `https://admin.ito4.jp:3002` にして `-Action up`。host-geoblock.sh の `ADMIN_PORTS` にも 3002 を戻す

### 5-5. nginx の設定が読めているか

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker exec km-nginx-proxy nginx -t

## 6. 戻し方

| 状況 | 戻し方 |
|---|---|
| DB に触らない像(nginx / certbot / postfix / mailpit)を上げた | compose の版を戻して `-Action up` |
| **Logto を上げて移行が走った** | **タグを戻すだけでは直らない。** 巻き戻し(下)か、控えからの復元 |
| MariaDB / PostgreSQL を上げた | 控えからの復元([03-backup](03-backup.ipynb) §8) |

**巻き戻しは万能ではない。** 列や表を落とす移行を戻しても、落ちた中身は返ってこない。確実なのは復元。
`PREV` を戻す先の版にしてから実行する。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto の DB 移行を巻き戻します(控えは取りましたか?)" --timeout 1800
PREV=1.43.0
docker exec km-logto sh -c "cd /etc/logto && npm run alteration rollback $PREV"

## 7. 上げたあとに見るもの

**healthy だけでは足りない**(中の HTTP が返ることしか言っていない)。

- [ ] `https://ito4.jp/` が開く
- [ ] 地図が出る。**教職員氏名の解除が効く**
- [ ] **管理画面にサインインできる**(= Logto が生きている)
- [ ] サインインのコードの画面で、1 つ目の枠に `autocomplete="one-time-code"` が付いている(nginx の差し込み。Logto が枠の作りを変えると黙って外れる。[04](04-host-jobs-and-mail.ipynb) §6)
- [ ] Android アプリでサインインできる
- [ ] 管理画面の「タスクマネージャー」の「サービス」タブで、全部が「稼働中」になる
- [ ] 開いておいた管理画面のタブに「新しい版を配備しました」の帯が出る(web のコードを変えたとき)
- [ ] ファイル管理から**実際にダウンロードできる**
- [ ] 問い合わせフォームから1通送って、**メールが届く**(送信側からは成功に見える)

## 8. コンテナを足す

### 8-1. 先に決めること

| 問い | なぜ |
|---|---|
| 外に出すのか | 出すなら **nginx 越し**。`ports:` を直に足さない |
| データを持つのか | ボリュームに **`name:` を必ず書く** |
| 秘密を持つのか | `.env` に置く。compose に直書きしない。**秘密に既定値を書かない** |
| 落ちたら何が止まるのか | `depends_on` の条件を決める材料 |

### 8-2. compose に書く(最低限そろえるもの)

```yaml
  たとえば-redis:
    image: redis:7.4-alpine@sha256:… # latest にしない。タグ@digest で固定する(check.php が見張る)
    container_name: km-redis         # km- で始める
    restart: unless-stopped
    security_opt:
      - no-new-privileges:true       # setuid で権限を上げる道を塞ぐ(mailserver のような例外は §1-1)
    mem_limit: 256m                  # 上限を必ず書く。1つの暴走でホストを道連れにしない
    pids_limit: 200
    read_only: true                  # イメージの中に書かせない。書く場所は tmpfs とボリューム(docs/12 §2)
    tmpfs:
      - /tmp
    cap_drop:
      - ALL                          # 要る特権だけ cap_add で戻す(check.php が見張る)
    expose:
      - "6379"                       # ports: ではなく expose:
    volumes:
      - redis_data:/data
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 30s
      timeout: 5s
      retries: 3
    networks:
      - edge                         # nginx から届く網。DB を使うなら appdb も。忘れると誰からも見えない
```

末尾の `volumes:` に:

```yaml
  redis_data:
    name: kosenmap_redis_data        # name: を必ず書く(check.php が見張る)
```

**`name:` が無いと**、プロジェクト名を変えた瞬間に compose は「無いので空を新規作成」して**起動に成功したまま中身が空になる。**
compose に `build:` を足したら、`check-updates.sh` の `BUILD_DOCKERFILES` にも足す。

### 8-3. 外に出すなら nginx を通す

`nginx/default.conf.template` の冒頭(upstream の並び)に `upstream km_example { zone km_up_example 64k; server たとえば-redis:6379 resolve; }` を足し、`location` では `proxy_pass http://km_example;` と書く。`server` の名前は**サービス名**(コンテナ名ではない)。
**`proxy_pass http://たとえば-redis:6379;` と直に書かない。** nginx は起動時に引いた IP を持ち続けるので、相手のコンテナを作り直すと IP が変わって **502** になる(2026-09-14 に Logto で発生)。`resolve` を付けると動いている間も引き直す。`check.php` が直書きを見つけて落ちる。
管理系にするなら `include /etc/nginx/km/allow-*.conf;` と `include /etc/nginx/km/gate-location.conf;` を **`proxy_pass` の手前**に置く。

### 8-4. 配備して確かめる

`check.php` → `-Action up -Backup`([02-deploy](02-deploy.ipynb))→ 下のセル → §7。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose ps
echo
docker volume ls --format '{{.Name}}' | sort

## 9. やってはいけないこと

| | なぜ |
|---|---|
| `image:` を `latest` にする | 誰も決めないまま版が上がる |
| ボリュームの `name:` を省く | 改名した瞬間、起動に成功したまま中身が空になる |
| 新しいサービスに `ports:` を足す | TLS・IP 制限・ログイン判定を迂回する口が増える。管理の画面は `127.0.0.1:` に出して SSH のトンネルで開く(Logto Console と同じ) |
| nginx の `proxy_pass` にコンテナ名を書く | 改名で壊れる。サービス名で書く |
| nginx で `proxy_pass http://サービス名:ポート;` と直に書く | 起動時の IP を持ち続け、相手を作り直すと 502。`upstream … resolve` を通す |
| Logto を控え無しで上げる | 移行は戻せない |
| `docker compose up -d <1つだけ>` | `depends_on` が評価されない |
| **`docker compose down -v`** | **ボリュームごと消す。** DB も証明書も DKIM 鍵も消える |
| 新しいサービスに `mem_limit` / `pids_limit` を書かない | 1つの暴走でホスト全体(RAM 2GB)が止まる |
| `mailserver` に `no-new-privileges` を付ける | `postdrop` が setgid で動くので送信が止まる(送信側からは成功に見える) |
| healthy を見て終わりにする | §7 を見る |

経緯と実測は [../Old/docs/containers.md](../Old/docs/containers.md)。

## 10. 緊急の修正版: PHP の RCE(CVE-2026-109618 / 109619。2026-10-21)

**何が起きたか**: PHP 本体に、認証なしで任意のコードを実行される穴が 2 つある(CVSS v4.0 9.2)。
修正版は **10/21 14:00 UTC(日本時間 23:00)**。詳細は予定より早く漏れ、攻撃の道具が出回っている。
弱い部品と対象の版は修正版と一緒に公表される(報道では multipart の受け取りが挙がっているが、確定ではない)。
**PHP はアプリのコードより前に本文を読む**ので、ログインの確認では防げない。本当の対策は PHP を上げることだけ。

このサイトで PHP が動くのは **web**(`php:8.4-apache`)と **phpmyadmin**(`phpmyadmin:5.2.3-apache`。IP の制限とゲートの奥)。
(**2026-11-02 に phpMyAdmin と Soketi を外した。** いまは PHP が動くのは web だけ。下の phpmyadmin・Soketi・テーブル管理の API キーの記述は当時の記録)

### 10-1. つなぎ(修正版を当てるまで)

| どこ | 何を | 外すか |
|---|---|---|
| nginx(`default.conf.template`) | `$km_php_shield_multipart`: multipart の要求を 443 と 8281 の server の直下で 403 | **10-22 に外した**(PHP 8.4.27 を当てて確かめた) |
| `docker/php/99-limits.ini` | `session.upload_progress.enabled = 0`(使っていない) | 残す |

止まる間は、ファイル管理・ポスター・階の画像・地図の取り込み・APK・プロフィールの画像・アプリのアカウント画像・パノラマ・AR・phpMyAdmin の取り込みが使えない。
配備は `.\deploy-to-host.ps1 -Backup -Yes` → `-Action restart -Services reverse-proxy` → `-Action restart -Services web`。
確かめ: `curl -s -o /dev/null -w '%{http_code}' -F a=b https://ito4.jp/` が 403、地図のページは 200。

### 10-1b. 修正版まで止めて待つ(利用者の判断。2026-10-21)

**入口(nginx)と証明書の更新(certbot)だけを動かし、ほかは止める。** web と Logto が居ない間は、nginx が
メンテナンス中の画面(`nginx/km/maintenance.html`。503)を返す。画面が出るのは、nginx 自身が出す 502(転送先が居ない)のときだけ
(`nginx/km/maintenance.conf`)。止める前に、配備して reverse-proxy を入れ直しておく(新しい画面の設定を読ませる)。

止める(PHP の 2 つを先に、DB はゆっくり止まれるよう 60 秒待つ):

```
ssh -i $env:USERPROFILE\.ssh\km_ops kmops@ito4.jp "docker stop -t 60 km-php-apache km-phpmyadmin; docker ps --format '{{.Names}}' | grep '^km-' | grep -vxE 'km-(nginx-proxy|certbot)' | xargs -r docker stop -t 60"
```

- `docker stop` で止めたコンテナは、`restart: unless-stopped` でも**自分では起き上がらない**。nginx を巻き込まない。
- 止めている間は、夜の控え(DB)が取れず、知らせのメールも出ない(どちらも止めているため)。止める前に `-Backup` で控えを取る。
- **止めている間に `-Action up` や `docker compose up -d` を打たない**(古い PHP のまま全部起き上がる)。ふつうの配備(`-Action` なし)は起こさない。
- 戻すのは **10-2 の `update-containers.ps1` の 2**(最後の `up -d` で全部が新しい PHP で起き上がる)。
  修正版を当てずに戻すときだけ、ホストで `docker compose up -d`。

### 10-2. 修正版を当てる

1. php.net の告知で、修正版(8.4 系)と弱い部品を読む。部品が multipart 以外なら、つなぎを見直す。
2. Docker の公式像が作り直されるのを待つ(普段は数時間〜1 日)。§4 の見張りが `php:8.4-apache` と `phpmyadmin:5.2.3-apache` の新しい中身を知らせる。
3. **§0 の `update-containers.ps1` の 2** で web(ビルド元)と phpmyadmin を選ぶ。
   止めたままでも当てられる(2026-10-22 に直した): 控えは止まっている DB を控えのあいだだけ起こして止め直し、止まっているコンテナも比べる。**直した版がホストに無い間は、先に `.\deploy-to-host.ps1 -Yes`(控えなし)で送る**(古い host-backup.sh は DB が止まっていると控えで止まる)。
   **★ が出ても修正版とは限らない**(8.4.26 のまま作り直されたものでも出る)。先に下の確かめで `php -v` を見る:
   `docker pull -q php:8.4-apache && docker run --rm --network none php:8.4-apache php -v`phpMyAdmin の像が遅れていても web を先に当てる(外に開いているのは web)。
4. 下の 🟢 のセルで `php -v` が修正版であることを確かめる。
5. **つなぎを外す**: nginx の map と 2 つの if、check.php の `php-rce-shield` の節を一緒に外す → 配備 → reverse-proxy の restart → ファイル管理で 1 つ上げて確かめる。
6. 手元の PHP(`Downloaded/php-8.4.24…`。試しの作りと check.php 用)も替え、古いものは Old/ へ。

### 10-3. 破られていないかを確かめる

穴は 10/20 から出回っていた。修正版を当てたら、下の 🟢 のセル(読むだけ)で見る。
src は読み取り専用で、web が書けるのは `uploads/`・`cache/`・設定の 2 ファイル・`/tmp` だけ。

- 書ける場所に、10/19 以降の `.php`・`.phtml`・`.htaccess`・実行できるファイルが無いこと。
- 設定の 2 ファイルに、値以外のコード(`eval`・`system`・`base64_decode` など)が無いこと。
- web の中のプロセスが apache2 だけであること(`sh`・`bash`・`curl`・`wget`・`nc`・`python` が居ない)。
- web の接続の相手が、Docker の網(172.16〜31.x)と 127.0.0.1 だけであること。
- nginx の POST の行き先に、見覚えのないものが無いこと。

**1 つでも怪しければ**、web が持つ秘密を全部取り替える(利用者)。
対象は MariaDB の Main、Logto のアプリと M2M 2 つ(Management API の権限あり)、Soketi、メール、reCAPTCHA、webhook の署名鍵、テーブル管理の API キー。
Logto の管理ログに知らない変更が無いかも見る。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
echo '--- 1. PHP の版(修正版であること)'
docker compose exec -T web php -v </dev/null | head -1
echo
echo '--- 2. 書ける場所の 10/19 以降のファイル(.php・.phtml・.htaccess・実行できるもの)'
find src/uploads src/cache -type f -newermt '2026-10-19' \( -name '*.php*' -o -name '*.phtml' -o -name '.htaccess' -o -perm -u+x \) -print 2>/dev/null
docker compose exec -T web find /tmp -type f -newermt '2026-10-19' \( -name '*.php*' -o -perm -u+x \) -print </dev/null 2>/dev/null
echo
echo '--- 3. 設定の 2 ファイルに値以外のコードが無いこと(何も出なければよい)'
grep -nE '\b(eval|system|exec|shell_exec|passthru|proc_open|popen|base64_decode|assert|include|require)\b|`' src/config/app-map.local.php src/config/map-access.local.php
echo
echo '--- 4. web の中のプロセス(apache2 だけのはず)'
docker top km-php-apache
echo
echo '--- 5. web の接続の相手(172.16〜31.x と 127.0.0.1 だけのはず)'
docker compose exec -T web cat /proc/net/tcp </dev/null | awk 'NR>1 && $4=="01" {print $3}' | sort -u | while IFS=: read -r h p; do
  printf '%d.%d.%d.%d:%d\n' "0x$(echo "$h" | cut -c7-8)" "0x$(echo "$h" | cut -c5-6)" "0x$(echo "$h" | cut -c3-4)" "0x$(echo "$h" | cut -c1-2)" "0x$p"
done
echo
echo '--- 6. nginx: 10/20 以降の POST の行き先と応答(多い順)'
docker compose logs --no-color --since 2026-10-20T00:00:00 reverse-proxy 2>/dev/null | grep -oE '"POST [^ ]+ [^"]+" [0-9]{3}' | sed -E 's/\?[^ ]*//' | sort | uniq -c | sort -rn | head -40